In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Görsel ayarlar
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_theme(style="whitegrid")

# 1. Dosya yollarını tanımlayalım
gen_path = "../data/raw/Plant_1_Generation_Data.csv"
weather_path = "../data/raw/Plant_1_Weather_Sensor_Data.csv"

# 2. Verileri okuyalım
df_gen = pd.read_csv(gen_path)
df_weather = pd.read_csv(weather_path)

print("Veriler başarıyla yüklendi!")


Veriler başarıyla yüklendi!


In [3]:
print("--- ÜRETİM VERİSİ ---")
display(df_gen.head())

print("\n--- HAVA DURUMU VERİSİ ---")
display(df_weather.head())

--- ÜRETİM VERİSİ ---


,DATE_TIME,PLANT_ID,SOURCE_KEY,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD
0,15-05-2020 00:00,4135001,1BY6WEcLGh8j5v7,0.0,0.0,0.0,6259559.0
1,15-05-2020 00:00,4135001,1IF53ai7Xc0U56Y,0.0,0.0,0.0,6183645.0
2,15-05-2020 00:00,4135001,3PZuoBAID5Wc2HD,0.0,0.0,0.0,6987759.0
3,15-05-2020 00:00,4135001,7JYdWkrLSPkdwr4,0.0,0.0,0.0,7602960.0
4,15-05-2020 00:00,4135001,McdE0feGgRqW7Ca,0.0,0.0,0.0,7158964.0



--- HAVA DURUMU VERİSİ ---


,DATE_TIME,PLANT_ID,SOURCE_KEY,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION
0,2020-05-15 00:00:00,4135001,HmiyD2TTLFNqkNe,25.184316,22.857507,0.0
1,2020-05-15 00:15:00,4135001,HmiyD2TTLFNqkNe,25.084589,22.761668,0.0
2,2020-05-15 00:30:00,4135001,HmiyD2TTLFNqkNe,24.935753,22.592306,0.0
3,2020-05-15 00:45:00,4135001,HmiyD2TTLFNqkNe,24.846130,22.360852,0.0
4,2020-05-15 01:00:00,4135001,HmiyD2TTLFNqkNe,24.621525,22.165423,0.0


In [4]:
print("--- ÜRETİM VERİSİ ÖZETİ ---")
print(f"Toplam Satır ve Sütun Sayısı: {df_gen.shape}")
print(df_gen.info())

print("\n--- HAVA DURUMU VERİSİ ÖZETİ ---")
print(f"Toplam Satır ve Sütun Sayısı: {df_weather.shape}")
print(df_weather.info())

--- ÜRETİM VERİSİ ÖZETİ ---
Toplam Satır ve Sütun Sayısı: (68778, 7)
<class 'pandas.DataFrame'>
RangeIndex: 68778 entries, 0 to 68777
Data columns (total 7 columns):
 #   Column       Non-Null Count  Dtype  
---  ------       --------------  -----  
 0   DATE_TIME    68778 non-null  str    
 1   PLANT_ID     68778 non-null  int64  
 2   SOURCE_KEY   68778 non-null  str    
 3   DC_POWER     68778 non-null  float64
 4   AC_POWER     68778 non-null  float64
 5   DAILY_YIELD  68778 non-null  float64
 6   TOTAL_YIELD  68778 non-null  float64
dtypes: float64(4), int64(1), str(2)
memory usage: 3.7 MB
None

--- HAVA DURUMU VERİSİ ÖZETİ ---
Toplam Satır ve Sütun Sayısı: (3182, 6)
<class 'pandas.DataFrame'>
RangeIndex: 3182 entries, 0 to 3181
Data columns (total 6 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   DATE_TIME            3182 non-null   str    
 1   PLANT_ID             3182 non-null   int64  
 2   SOURCE_KEY         

In [5]:
import pandas as pd

# ---------------------------------------------------------
# 1. Tarih-Saat Dönüşümü
# ---------------------------------------------------------
# Veri setindeki tarih formatı genellikle 'dd-mm-YYYY HH:MM' veya 'YYYY-mm-dd HH:MM' olabilir.
# dayfirst=True parametresi gg-aa-yyyy formatını güvenle çözer.
df_gen['DATE_TIME'] = pd.to_datetime(df_gen['DATE_TIME'], dayfirst=True)
df_weather['DATE_TIME'] = pd.to_datetime(df_weather['DATE_TIME'], dayfirst=True)

# ---------------------------------------------------------
# 2. İnvertör Üretimlerini Tesis Bazında Toplama (Agregasyon)
# ---------------------------------------------------------
# Bir tesiste birden fazla invertör aynı zaman diliminde güç üretir.
# DC_POWER ve AC_POWER değerlerini topluyoruz; DAILY_YIELD ve TOTAL_YIELD için de toplam alıyoruz.
df_gen_aggregated = df_gen.groupby('DATE_TIME').agg({
    'DC_POWER': 'sum',
    'AC_POWER': 'sum',
    'DAILY_YIELD': 'sum',
    'TOTAL_YIELD': 'sum'
}).reset_index()

# ---------------------------------------------------------
# 3. Hava Durumu Verisini Hazırlama
# ---------------------------------------------------------
# df_weather tablosunda gereksiz veya tekil olan sütunları filtreleyip sadeleştirelim
weather_cols = ['DATE_TIME', 'AMBIENT_TEMPERATURE', 'MODULE_TEMPERATURE', 'IRRADIATION']
df_weather_clean = df_weather[weather_cols].copy()

# Olası mükerrer (duplicate) zaman damgalarını temizleyelim
df_weather_clean = df_weather_clean.drop_duplicates(subset=['DATE_TIME'])

# ---------------------------------------------------------
# 4. Üretim ve Hava Durumu Verilerini Birleştirme (Merge)
# ---------------------------------------------------------
# Her iki veri kümesinin kesişimini alıyoruz (inner join)
df_merged = pd.merge(df_gen_aggregated, df_weather_clean, on='DATE_TIME', how='inner')

# Zaman serisi analizinde sıralama hayati önem taşır:
df_merged = df_merged.sort_values('DATE_TIME').reset_index(drop=True)

# ---------------------------------------------------------
# 5. Kontrol ve Doğrulama
# ---------------------------------------------------------
print("--- Özet Bilgiler ---")
print(f"Orijinal Üretim Kayıt Sayısı      : {len(df_gen):,}")
print(f"Toplulaştırılmış Üretim Zaman Adımı: {len(df_gen_aggregated):,}")
print(f"Hava Durumu Zaman Adımı           : {len(df_weather_clean):,}")
print(f"Birleştirilmiş Nihai Satır Sayısı  : {len(df_merged):,}")
print("\nEksik Değer Durumu:")
print(df_merged.isnull().sum())
print("\nİlk 5 Satır:")
display(df_merged.head())

--- Özet Bilgiler ---
Orijinal Üretim Kayıt Sayısı      : 68,778
Toplulaştırılmış Üretim Zaman Adımı: 3,158
Hava Durumu Zaman Adımı           : 3,182
Birleştirilmiş Nihai Satır Sayısı  : 3,157

Eksik Değer Durumu:
DATE_TIME              0
DC_POWER               0
AC_POWER               0
DAILY_YIELD            0
TOTAL_YIELD            0
AMBIENT_TEMPERATURE    0
MODULE_TEMPERATURE     0
IRRADIATION            0
dtype: int64

İlk 5 Satır:


C:\Users\LENOVO\AppData\Local\Temp\ipykernel_4528\4193938753.py:9: UserWarning: Parsing dates in %Y-%m-%d %H:%M:%S format when dayfirst=True was specified. Pass `dayfirst=False` or specify a format to silence this warning.
  df_weather['DATE_TIME'] = pd.to_datetime(df_weather['DATE_TIME'], dayfirst=True)


,DATE_TIME,DC_POWER,AC_POWER,DAILY_YIELD,TOTAL_YIELD,AMBIENT_TEMPERATURE,MODULE_TEMPERATURE,IRRADIATION
0,2020-05-15 00:00:00,0.0,0.0,0.0,143581676.0,25.184316,22.857507,0.0
1,2020-05-15 00:15:00,0.0,0.0,0.0,143581676.0,25.084589,22.761668,0.0
2,2020-05-15 00:30:00,0.0,0.0,0.0,143581676.0,24.935753,22.592306,0.0
3,2020-05-15 00:45:00,0.0,0.0,0.0,143581676.0,24.846130,22.360852,0.0
4,2020-05-15 01:00:00,0.0,0.0,0.0,150761642.0,24.621525,22.165423,0.0


In [6]:
# 1. Zaman aralıklarının kontrolü (farkların dağılımı)
time_diffs = df_merged['DATE_TIME'].diff()
print("Zaman Adımları Arasındaki Farklar:")
print(time_diffs.value_counts())

# 2. Temizlenmiş ve birleştirilmiş veriyi processed klasörüne kaydetme
df_merged.to_csv('../data/processed/df_merged.csv', index=False)
print("\n'data/processed/df_merged.csv' başarıyla kaydedildi.")

Zaman Adımları Arasındaki Farklar:
DATE_TIME
0 days 00:15:00    3146
0 days 00:30:00       3
0 days 03:00:00       1
0 days 01:00:00       1
0 days 04:15:00       1
0 days 09:00:00       1
0 days 01:45:00       1
0 days 08:00:00       1
0 days 00:45:00       1
Name: count, dtype: int64

'data/processed/df_merged.csv' başarıyla kaydedildi.
